# 12.5 一小段聲音有哪些頻率？

# 第 12 章：沿用相同介面加入聲音

前置：第10章模態展開與第7章loss mask。頻譜可獨立讀；聲音生成與log-mel都從PyTorch寫起，不依賴Whisper或torchaudio。FSDD爲額外真實語音pilot；本章默認合成音高／時長任務。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：從聽到的起伏，找到每一小段的頻率**

frame→window→STFT→mel→log，每次只加一個步驟。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/audio.svg")))

**先想一想：**16kHz、n_fft400，每格相差40Hz還是400Hz？

STFT每框做Fourier變換：問這一小段含哪些頻率，再沿時間擺成圖。複數同時保存幅度與相位，先看功率。

**把直覺寫成數學：**frequency_bin≈k·sample_rate/n_fft；frequency分辨率≈sample_rate/n_fft。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.multimodal import tone

wave = tone(440.0, seconds=0.2)
spectrum = torch.stft(wave, 400, 160, window=torch.hann_window(400), return_complex=True)
power = spectrum.abs().square()
peak = power.mean(-1).argmax().item()
print(power.shape, "peak Hz", peak * 16000 / 400)
assert abs(peak * 40 - 440) <= 40

**如何讀結果：**一秒內不是隻有一個頻率格；有限框與window會使峯附近也有能量。

**只改一件事：**只換成880Hz，覈對峯位置。
